# Embedding a Knowledge Assistant in Microsoft Teams

This notebook walks through the production-ready way to put a Databricks Knowledge Assistant in front of Teams users: deploy a small Streamlit chat UI as a Databricks App, then add it as a tab in a Teams channel.

The notebook is written for someone new to Databricks. Every step is explicit. Do not skip any of them.

### Why a Databricks App and not just the Knowledge Assistant URL directly

The Knowledge Assistant agent in Agent Bricks has a built-in chat URL inside Databricks, and that URL can technically be added as a Teams Website tab. That path is fine for a quick internal demo, but it has three real limitations:

- Every Teams user needs a Databricks workspace login. First-time users get a Databricks SSO prompt inside the Teams tab.
- The Databricks workspace navigation appears inside the Teams tab. The experience looks like "Databricks running inside Teams" instead of a purpose-built chatbot.
- There is no branding control. You cannot change colors, add a logo, or customize the empty-state message.

The Databricks App approach removes all three of those limitations. The App runs as a service principal so end users do not need individual Databricks accounts. The UI is a clean Streamlit chat panel with no Databricks chrome. Branding is fully customizable. This is the right path for any production-facing rollout.

Time required: about 45-60 minutes for the first deploy. Subsequent updates take minutes.

## Prerequisites

Before starting, confirm you have:

- A deployed Knowledge Assistant agent in Databricks Agent Bricks. If you do not have one yet, follow the [Knowledge Assistant docs](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant) first. The agent must show a serving endpoint name on its detail page.
- A Databricks workspace with Apps enabled. If you cannot see **Compute** > **Apps** in the left sidebar, ask your Databricks workspace administrator to enable the Apps feature.
- A Microsoft Teams workspace where you have permission to add tabs to channels. If you are not sure, the easiest check is to open Teams, navigate to a channel, and look for a `+` button at the top of the channel for "Add a tab." If you see it, you have permission.

## Placeholders used in this notebook

The notebook uses placeholders that you must replace with values from your environment. They all appear in angle brackets:

| Placeholder | Description | Where to get it |
|---|---|---|
| `<WORKSPACE_URL>` | Your Databricks workspace URL | See Setup section below |
| `<DATABRICKS_TOKEN>` | A personal access token for testing | See Setup section below |
| `<KA_ENDPOINT_NAME>` | The Knowledge Assistant serving endpoint name | The agent's detail page in Agent Bricks |
| `<APP_NAME>` | A name you choose for the Databricks App | You pick this (e.g., `ka-chat`) |

## Setup: get your workspace URL and a personal access token

Two values are needed before you can run the test query in Step 1. Both are easy to find in the Databricks UI.

### A. Find your workspace URL

The workspace URL is the address of your Databricks workspace as it appears in your browser. To find it:

1. Open your Databricks workspace in a browser.
2. Look at the browser's address bar.
3. Copy everything from `https://` up to and including `.databricks.com`. Do not include any path after that.

Examples of what a workspace URL looks like:

```
https://e2-demo-field-eng.cloud.databricks.com
https://acme-corp.cloud.databricks.com
https://adb-1234567890.18.azuredatabricks.net
```

Save this value. You will paste it in as `<WORKSPACE_URL>`.

### B. Create a personal access token (PAT)

A personal access token is a string of characters that authenticates API calls as you. It is the password Databricks uses for programmatic access. You only need this for the one-time test query in Step 1; the deployed App will use its own service principal credentials (more on that in Step 4).

To generate one:

1. In Databricks, click your user avatar in the top-right corner.
2. Click **Settings**.
3. In the left sidebar, click **Developer**.
4. Under **Access tokens**, click **Manage**.
5. Click **Generate new token**.
6. Set:
   - **Comment**: anything memorable, e.g., `Notebook 03 test query`
   - **Lifetime (days)**: 30 is reasonable for a one-time test
7. Click **Generate**.
8. **Copy the token immediately.** Databricks will display it once. If you click away, it is gone forever and you have to generate a new one.

Save this value somewhere temporary (a password manager or a scratch text file you will discard after testing). You will paste it in as `<DATABRICKS_TOKEN>`.

**Security note:** never commit a notebook with a real token in it to a Git repository. After running Step 1, either remove the token from the notebook or delete the token in Databricks.

## Step 1: Verify the Knowledge Assistant endpoint is queryable

Before building any UI, confirm the endpoint actually responds to a question. This avoids debugging Teams or App configuration when the real problem is on the Databricks serving side.

Replace the three placeholder values below with the values you collected in Setup, then run the cell. A successful response (status 200 with an answer body) confirms the endpoint is healthy and the conversation API works.

In [0]:
import requests

WORKSPACE_URL    = '<WORKSPACE_URL>'      # e.g., https://e2-demo-field-eng.cloud.databricks.com
KA_ENDPOINT_NAME = '<KA_ENDPOINT_NAME>'   # the name from the Agent Bricks agent page
TOKEN            = '<DATABRICKS_TOKEN>'   # the PAT you generated in Setup B

response = requests.post(
    f'{WORKSPACE_URL.rstrip("/")}/serving-endpoints/{KA_ENDPOINT_NAME}/invocations',
    headers={'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'},
    json={
        'messages': [
            {'role': 'user', 'content': 'What is this assistant for?'}
        ]
    },
    timeout=60,
)

print('Status:', response.status_code)
print('Body:', response.json())

## Step 2: Build the chat UI

The chat UI is a small Streamlit app whose only job is to talk to the Knowledge Assistant serving endpoint. The code below:

- Streams responses token-by-token using `st.write_stream` so users see text appear as it is generated. No waiting on the full response.
- Reuses a single HTTP connection across requests for low per-message latency.
- Hides the Streamlit menu, footer, and header so the UI looks clean inside the Teams tab.
- Handles errors gracefully with user-friendly messages instead of stack traces.
- Maintains conversation history within a session.

Create a folder anywhere in your Databricks workspace (e.g., `/Workspace/Users/<your-email>/ka-chat-app`). Save the Python code below as `app.py` in that folder. The next markdown cell covers the two supporting files that go alongside it.

### About the environment variables this code reads

The `app.py` code reads three environment variables: `DATABRICKS_HOST`, `DATABRICKS_TOKEN`, and `KA_ENDPOINT_NAME`. You do not set these in `app.py`. You will not set most of them anywhere by hand:

- `DATABRICKS_HOST` is **automatically injected by the Databricks Apps runtime when the App runs**. You do not configure this.
- `DATABRICKS_TOKEN` is also **automatically injected by the Databricks Apps runtime**. The value is an OAuth token for the App's auto-created service principal (see Step 4). You do not configure this.
- `KA_ENDPOINT_NAME` is the only one you specify. You set it in `app.yaml` (see the next section).

In [0]:
# app.py
# Minimal, fast, clean chat UI that calls a Knowledge Assistant serving endpoint.

import json
import os

import requests
import streamlit as st

WORKSPACE_URL    = os.environ['DATABRICKS_HOST'].rstrip('/')
KA_ENDPOINT_NAME = os.environ['KA_ENDPOINT_NAME']
TOKEN            = os.environ['DATABRICKS_TOKEN']

st.set_page_config(
    page_title='Knowledge Assistant',
    layout='centered',
    initial_sidebar_state='collapsed',
)

# Hide Streamlit chrome for a clean Teams-friendly look
st.markdown(
    '''
    <style>
      #MainMenu, footer, header { visibility: hidden; }
      .block-container { padding-top: 1rem; padding-bottom: 1rem; }
      [data-testid="stChatMessage"] { padding: 0.5rem 1rem; }
    </style>
    ''',
    unsafe_allow_html=True,
)

# Reused HTTP session for connection pooling (lower latency per turn)
if 'http' not in st.session_state:
    sess = requests.Session()
    sess.headers.update({
        'Authorization': f'Bearer {TOKEN}',
        'Content-Type': 'application/json',
    })
    st.session_state.http = sess

if 'messages' not in st.session_state:
    st.session_state.messages = []

st.title('Knowledge Assistant')

# Render conversation history
for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        st.markdown(msg['content'])


def stream_response(messages):
    """Stream tokens from the Knowledge Assistant serving endpoint."""
    url = f'{WORKSPACE_URL}/serving-endpoints/{KA_ENDPOINT_NAME}/invocations'
    payload = {'messages': messages, 'stream': True}

    with st.session_state.http.post(url, json=payload, stream=True, timeout=120) as resp:
        if resp.status_code != 200:
            yield f'Error from Knowledge Assistant: HTTP {resp.status_code}'
            return

        for line in resp.iter_lines(decode_unicode=True):
            if not line or not line.startswith('data: '):
                continue
            data_str = line[6:]
            if data_str == '[DONE]':
                break
            try:
                chunk = json.loads(data_str)
                delta = chunk.get('choices', [{}])[0].get('delta', {}).get('content', '')
                if delta:
                    yield delta
            except json.JSONDecodeError:
                continue


# Handle new user input
if prompt := st.chat_input('Ask a question'):
    st.session_state.messages.append({'role': 'user', 'content': prompt})
    with st.chat_message('user'):
        st.markdown(prompt)

    with st.chat_message('assistant'):
        try:
            answer = st.write_stream(stream_response(st.session_state.messages))
        except requests.exceptions.Timeout:
            answer = 'The request took too long. Please try a more focused question.'
            st.error(answer)
        except Exception as exc:
            answer = f'Something went wrong: {str(exc)[:200]}'
            st.error(answer)
        st.session_state.messages.append({'role': 'assistant', 'content': answer})

### Supporting files

Save the following two files alongside `app.py` in the same folder. These tell Databricks how to run your App.

**`app.yaml`** - configures the App and sets the one environment variable you control:

```yaml
command: ['streamlit', 'run', 'app.py', '--server.port', '8080', '--server.address', '0.0.0.0', '--server.headless', 'true']

env:
  - name: KA_ENDPOINT_NAME
    value: '<KA_ENDPOINT_NAME>'
```

Replace `<KA_ENDPOINT_NAME>` with the Knowledge Assistant serving endpoint name from Agent Bricks. You do NOT need to set `DATABRICKS_HOST` or `DATABRICKS_TOKEN` here. The Databricks Apps runtime injects those automatically.

**`requirements.txt`** - lists Python packages the App needs:

```text
streamlit==1.39.0
requests==2.32.3
```

## Step 3: Deploy the App

At this point you should have three files in a workspace folder: `app.py`, `app.yaml`, and `requirements.txt`. Now deploy them as a running Databricks App.

1. In Databricks, click **Compute** > **Apps** in the left sidebar.
2. Click **Create app**.
3. Set:
   - **Name**: a name of your choice (e.g., `ka-chat`). This becomes part of the App URL and the name of the auto-created service principal.
   - Leave other settings at defaults.
4. Click **Create**.
5. On the App's detail page, click **Deploy**.
6. Choose the workspace folder where you saved your three files.
7. Click **Deploy** to confirm.
8. Wait 1-2 minutes for the status to change from "Deploying" to "Running."

Once running, the App's detail page shows its URL at the top:

```
https://<APP_NAME>-<workspace-id>.<region>.databricksapps.com
```

Copy this URL. Open it in a new browser tab to verify the chat UI loads. You will see a permission error if you try to chat right now. That is expected and resolved in Step 4.

### Performance: keep the app warm

By default, Databricks Apps scale to zero when idle. The first request after idle hits a cold start of 10-30 seconds. For a production user-facing chatbot this is too slow.

Two options:

1. Configure the App's compute to stay always-on. In the app's Compute settings, set the minimum instances to 1. Slightly higher cost, no cold starts.
2. Schedule a cheap keep-alive ping every 5 minutes. Set up a Databricks Job that runs `curl https://<your-app-url>/healthz` on a 5-minute cron. Free, but adds a small orchestration dependency.

Pick option 1 for production. Use option 2 only if cost is a critical constraint.

## Step 4: Grant the App's service principal permission to call the Knowledge Assistant

### What is a service principal and where does it come from

A service principal is a non-human identity that an application uses to authenticate to Databricks. Think of it as a "robot user" with its own ID and its own permissions, separate from any human user.

**You do not need to manually create a service principal for this App.** When you deployed the App in Step 3, Databricks automatically created one for it. The auto-created service principal has a name that matches `<APP_NAME>-app`. So if you named your App `ka-chat`, the service principal is named `ka-chat-app`. You can see it in **Settings** > **Identity and access** > **Service principals**.

Databricks Apps automatically inject the service principal's OAuth token into the running App as the `DATABRICKS_TOKEN` environment variable. That is why `app.py` can authenticate to the Knowledge Assistant endpoint without you ever handling a token directly. The App authenticates as the service principal, not as any individual user.

### Granting CAN_QUERY

The Knowledge Assistant serving endpoint controls who can query it via permissions. The App's auto-created service principal does not have any permissions by default. You need to explicitly grant it the `CAN_QUERY` permission on the Knowledge Assistant endpoint, or every request will return HTTP 403.

1. In Databricks, navigate to **Compute** > **Serving**.
2. Find the Knowledge Assistant endpoint (named `<KA_ENDPOINT_NAME>`).
3. Click the endpoint to open its detail page.
4. Click the **Permissions** tab.
5. Click **Grant**.
6. In the grantee picker, search for `<APP_NAME>-app` (the auto-created service principal). Select it.
7. Check the `CAN_QUERY` box.
8. Click **Grant**.

The App can now successfully call the endpoint. Go back to the App's URL in your browser and try asking a question. The chat should now respond.

## Step 5: Add the App as a Teams tab

1. Open Microsoft Teams.
2. Navigate to the channel where the Knowledge Assistant should live (e.g., a Data Governance or Compliance channel).
3. At the top of the channel, click **+ (Add a tab)**.
4. Search for and select **Website**.
5. In the configuration dialog, set:
   - **Tab name**: `Knowledge Assistant`
   - **URL**: the Databricks App URL from Step 3 (looks like `https://<APP_NAME>-<workspace-id>.<region>.databricksapps.com`)
6. Click **Save**.

The chat UI now appears as a Teams tab. Clean branding, no Databricks chrome, no per-user Databricks login prompt.

## Step 6: Test the end-user experience

1. From a colleague's Teams account (or your own using a different identity), open the channel.
2. Click the Knowledge Assistant tab.
3. Confirm the chat loads quickly (sub-second if the App is warm).
4. Ask three to five real questions. Confirm:
   - Responses stream in token-by-token, not as one big block.
   - Citations and source references appear correctly.
   - Conversation history persists across messages within a session.
   - Refreshing the tab clears the conversation (Streamlit session resets, expected behavior).

### Troubleshooting

- **Chat returns "Error from Knowledge Assistant: HTTP 403":** the service principal does not have `CAN_QUERY` on the endpoint. Re-do Step 4 carefully.
- **First message is very slow (10-30 seconds), subsequent messages are fast:** the App scaled to zero. See Step 3's keep-warm options.
- **Page does not load at all in the Teams tab:** confirm the App is in "Running" state in **Compute** > **Apps**, and that the URL you pasted into the Teams tab matches the App's URL exactly.
- **The chat appears but answers are wrong or empty:** the Knowledge Assistant endpoint is responding but its corpus is not what you expect. Open the agent in Agent Bricks and verify the indexed documents are correct.

## Heavier alternatives, for completeness

The Databricks App + Teams tab pattern above covers the vast majority of use cases. Two more sophisticated alternatives exist if a specific limitation forces it:

- **Native Microsoft Teams Bot (Bot Framework SDK).** Users can `@mention` the bot in any channel or DM, not just inside a specific channel tab. Requires Microsoft Bot Framework SDK, Azure Bot Service registration, and Azure hosting (Functions or App Service). Multi-week development effort. Use when stakeholders demand chat-anywhere instead of chat-in-a-tab.
- **Microsoft Copilot Studio custom skill.** Wraps the Knowledge Assistant endpoint as a Copilot skill, published to Teams via Copilot Studio. Adds Copilot Studio licensing. Use when the organization already invests in Copilot Studio and wants centralized agent governance there.

Both alternatives still point at the same Knowledge Assistant serving endpoint. Only the consumption layer changes.

## References

- [Knowledge Assistant](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant)
- [Databricks Apps overview](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/)
- [Streamlit in Databricks Apps](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/get-started/streamlit)
- [Service principals in Databricks](https://docs.databricks.com/aws/en/admin/users-groups/service-principals)
- [Model Serving endpoint permissions](https://docs.databricks.com/aws/en/machine-learning/model-serving/manage-serving-endpoints)
- [Personal access tokens](https://docs.databricks.com/aws/en/dev-tools/auth/pat)
- [Adding Website tabs in Microsoft Teams](https://learn.microsoft.com/en-us/microsoftteams/manage-website-tab)